# Hubble.AI Treasury Cash-Flow Forecasting - End-to-End Notebook

**Purpose**: Complete pipeline from raw data to model evaluation for W1-W8 cash-flow forecasting.

**Target WAPE by Horizon**:
- W1: ≤ 5.0%
- W2: ≤ 7.5%
- W3: ≤ 10.0%
- W4: ≤ 12.5%
- W5: ≤ 15.0%
- W6: ≤ 17.5%
- W7: ≤ 20.0%
- W8: ≤ 22.5%

**Treasury LP Baseline**: ~10% W1 WAPE

**Current Best ML Result**: 19.79% W1 WAPE (LightGBM Recursive, Enhanced Dataset)

## 1. Setup and Dependencies

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# ML libraries
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

# Stats
from scipy import stats
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Project modules
import sys
sys.path.append('../src')

print("Dependencies loaded successfully")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"LightGBM: {lgb.__version__}")
print(f"XGBoost: {xgb.__version__}")

## 2. Load Raw Data

We have two primary data sources:
1. **Actuals**: Daily transaction data (needs weekly aggregation)
2. **LP Forecasts**: Treasury's manual forecasts (W1-W4 horizons)

In [ ]:
# Load actuals (daily transaction data)
actuals_df = pd.read_csv('../data/raw/actuals.csv')
actuals_df['posting_date'] = pd.to_datetime(actuals_df['posting_date'])

print("=" * 80)
print("ACTUALS DATA (Raw Daily Transactions)")
print("=" * 80)
print(f"Shape: {actuals_df.shape}")
print(f"Date range: {actuals_df['posting_date'].min()} to {actuals_df['posting_date'].max()}")
print(f"Entities: {actuals_df['entity_id'].nunique()}")
print(f"Liquidity groups: {actuals_df['liquidity_group'].nunique()}")
print(f"\nColumns: {list(actuals_df.columns)}")
print("\nFirst 5 rows:")
actuals_df.head()

In [ ]:
# Load LP forecasts (Treasury manual forecasts)
lp_df = pd.read_csv('../data/raw/lp_forecasts.csv')
lp_df['forecast_date'] = pd.to_datetime(lp_df['forecast_date'])

print("=" * 80)
print("LP FORECASTS DATA (Treasury Manual Forecasts)")
print("=" * 80)
print(f"Shape: {lp_df.shape}")
print(f"Date range: {lp_df['forecast_date'].min()} to {lp_df['forecast_date'].max()}")
print(f"Entities: {lp_df['entity_id'].nunique()}")
print(f"Liquidity groups: {lp_df['liquidity_group'].nunique()}")
print(f"\nColumns: {list(lp_df.columns)}")
print("\nFirst 5 rows:")
lp_df.head()

## 3. Initial EDA - Raw Data

In [ ]:
# Actuals: Missing values
print("Actuals - Missing Values:")
print(actuals_df.isnull().sum())
print("\nActuals - Data Types:")
print(actuals_df.dtypes)
print("\nActuals - Summary Statistics:")
actuals_df.describe()

In [ ]:
# LP Forecasts: Missing values
print("LP Forecasts - Missing Values:")
print(lp_df.isnull().sum())
print("\nLP Forecasts - Data Types:")
print(lp_df.dtypes)
print("\nLP Forecasts - Summary Statistics:")
lp_df.describe()

In [ ]:
# Visualize actuals distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Amount distribution
axes[0].hist(actuals_df['amount_eur'], bins=50, edgecolor='black')
axes[0].set_xlabel('Amount (EUR)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Daily Amounts')
axes[0].axvline(0, color='red', linestyle='--', label='Zero line')
axes[0].legend()

# Transactions over time
daily_agg = actuals_df.groupby('posting_date')['amount_eur'].sum()
axes[1].plot(daily_agg.index, daily_agg.values)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Total Amount (EUR)')
axes[1].set_title('Total Daily Cash Flow Over Time')
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 4. Data Cleaning

In [ ]:
# Clean actuals
print("Cleaning Actuals Data...")
actuals_clean = actuals_df.copy()

# Remove duplicates
before_dedup = len(actuals_clean)
actuals_clean = actuals_clean.drop_duplicates()
after_dedup = len(actuals_clean)
print(f"  Removed {before_dedup - after_dedup} duplicate rows")

# Handle missing values
print(f"  Missing values before cleaning: {actuals_clean.isnull().sum().sum()}")
actuals_clean = actuals_clean.dropna(subset=['entity_id', 'liquidity_group', 'posting_date', 'amount_eur'])
print(f"  Missing values after cleaning: {actuals_clean.isnull().sum().sum()}")

print(f"\nFinal actuals shape: {actuals_clean.shape}")

In [ ]:
# Clean LP forecasts
print("Cleaning LP Forecasts Data...")
lp_clean = lp_df.copy()

# Remove duplicates
before_dedup = len(lp_clean)
lp_clean = lp_clean.drop_duplicates()
after_dedup = len(lp_clean)
print(f"  Removed {before_dedup - after_dedup} duplicate rows")

# Handle missing values
print(f"  Missing values before cleaning: {lp_clean.isnull().sum().sum()}")
lp_clean = lp_clean.dropna(subset=['entity_id', 'liquidity_group', 'forecast_date'])
print(f"  Missing values after cleaning: {lp_clean.isnull().sum().sum()}")

print(f"\nFinal LP forecasts shape: {lp_clean.shape}")

## 5. Daily-Level Feature Engineering (BEFORE Aggregation)

**CRITICAL**: Extract daily patterns BEFORE aggregating to weekly level.

Daily features to capture:
- Day of week patterns
- Weekend vs weekday
- Position in month (early/mid/late)
- Transaction timing and concentration

In [ ]:
# Add daily-level features to actuals_clean
print("Creating Daily-Level Features...")
print(f"Starting shape: {actuals_clean.shape}")
print()

# Day of week features
actuals_clean['day_of_week'] = actuals_clean['posting_date'].dt.dayofweek  # 0=Monday
actuals_clean['is_weekend'] = actuals_clean['day_of_week'].isin([5, 6]).astype(int)
actuals_clean['is_monday'] = (actuals_clean['day_of_week'] == 0).astype(int)
actuals_clean['is_friday'] = (actuals_clean['day_of_week'] == 4).astype(int)

# Position in month
actuals_clean['day_of_month'] = actuals_clean['posting_date'].dt.day
actuals_clean['is_month_start'] = (actuals_clean['day_of_month'] <= 5).astype(int)
actuals_clean['is_month_mid'] = ((actuals_clean['day_of_month'] > 10) & (actuals_clean['day_of_month'] <= 20)).astype(int)
actuals_clean['is_month_end'] = (actuals_clean['day_of_month'] >= 25).astype(int)

# Position in week (for within-week patterns)
actuals_clean['week_start_daily'] = actuals_clean['posting_date'] - pd.to_timedelta(
    actuals_clean['posting_date'].dt.dayofweek, unit='D'
)
actuals_clean['days_from_week_start'] = (actuals_clean['posting_date'] - actuals_clean['week_start_daily']).dt.days
actuals_clean['is_early_week'] = (actuals_clean['days_from_week_start'] <= 2).astype(int)
actuals_clean['is_late_week'] = (actuals_clean['days_from_week_start'] >= 4).astype(int)

# Quarter flags
actuals_clean['month'] = actuals_clean['posting_date'].dt.month
actuals_clean['is_quarter_end_month'] = actuals_clean['month'].isin([3, 6, 9, 12]).astype(int)

print("Daily-level features created:")
daily_features = ['day_of_week', 'is_weekend', 'is_monday', 'is_friday',
                  'day_of_month', 'is_month_start', 'is_month_mid', 'is_month_end',
                  'days_from_week_start', 'is_early_week', 'is_late_week',
                  'is_quarter_end_month']
for feat in daily_features:
    print(f"  - {feat}")
print()
print(f"Final shape: {actuals_clean.shape}")

In [ ]:
# Visualize daily patterns
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Day of week pattern
dow_pattern = actuals_clean.groupby('day_of_week')['amount_eur'].agg(['mean', 'sum', 'count'])
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[0, 0].bar(range(7), dow_pattern['mean'])
axes[0, 0].set_xticks(range(7))
axes[0, 0].set_xticklabels(dow_labels)
axes[0, 0].set_xlabel('Day of Week')
axes[0, 0].set_ylabel('Mean Amount (EUR)')
axes[0, 0].set_title('Average Daily Amount by Day of Week')
axes[0, 0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0, 0].grid(True, alpha=0.3)

# Weekend vs Weekday
weekend_pattern = actuals_clean.groupby('is_weekend')['amount_eur'].agg(['mean', 'sum', 'count'])
axes[0, 1].bar(['Weekday', 'Weekend'], weekend_pattern['mean'])
axes[0, 1].set_ylabel('Mean Amount (EUR)')
axes[0, 1].set_title('Average Daily Amount: Weekday vs Weekend')
axes[0, 1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0, 1].grid(True, alpha=0.3)

# Position in month
dom_pattern = actuals_clean.groupby('day_of_month')['amount_eur'].mean()
axes[1, 0].plot(dom_pattern.index, dom_pattern.values, marker='o', markersize=4)
axes[1, 0].set_xlabel('Day of Month')
axes[1, 0].set_ylabel('Mean Amount (EUR)')
axes[1, 0].set_title('Average Daily Amount by Day of Month')
axes[1, 0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1, 0].axvline(5, color='green', linestyle='--', alpha=0.3, label='Early month')
axes[1, 0].axvline(25, color='orange', linestyle='--', alpha=0.3, label='Late month')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Transaction count by day of week
axes[1, 1].bar(range(7), dow_pattern['count'])
axes[1, 1].set_xticks(range(7))
axes[1, 1].set_xticklabels(dow_labels)
axes[1, 1].set_xlabel('Day of Week')
axes[1, 1].set_ylabel('Transaction Count')
axes[1, 1].set_title('Transaction Count by Day of Week')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print("Day of Week Pattern:")
print(dow_pattern)
print()
print("Weekend vs Weekday Pattern:")
print(weekend_pattern)

## 6. Aggregate Actuals to WeeklyConvert daily transactions to weekly aggregates (Monday-Sunday weeks).

In [ ]:
# Create week_start column (Monday of each week)actuals_clean['week_start'] = actuals_clean['posting_date'] - pd.to_timedelta(    actuals_clean['posting_date'].dt.dayofweek, unit='D')# Aggregate to weekly WITH daily pattern statisticsprint("Aggregating daily data to weekly WITH daily pattern features...")weekly_actuals = actuals_clean.groupby(    ['entity_id', 'liquidity_group', 'week_start']).agg({    'amount_eur': 'sum',    'posting_date': 'count',  # Transaction count    # Daily pattern aggregations    'is_weekend': 'sum',  # Number of weekend days with transactions    'is_monday': 'sum',    'is_friday': 'sum',    'is_month_start': 'sum',    'is_month_end': 'sum',    'is_early_week': 'sum',    'is_late_week': 'sum'}).rename(columns={    'posting_date': 'transaction_count',    'is_weekend': 'weekend_transaction_days',    'is_monday': 'monday_transactions',    'is_friday': 'friday_transactions',    'is_month_start': 'month_start_days',    'is_month_end': 'month_end_days',    'is_early_week': 'early_week_days',    'is_late_week': 'late_week_days'}).reset_index()# Add percentage featuresweekly_actuals['pct_weekend_days'] = weekly_actuals['weekend_transaction_days'] / weekly_actuals['transaction_count']weekly_actuals['pct_early_week'] = weekly_actuals['early_week_days'] / weekly_actuals['transaction_count']weekly_actuals['pct_late_week'] = weekly_actuals['late_week_days'] / weekly_actuals['transaction_count']print("=" * 80)print("WEEKLY ACTUALS (WITH DAILY PATTERN FEATURES)")print("=" * 80)print(f"Shape: {weekly_actuals.shape}")print(f"Date range: {weekly_actuals['week_start'].min()} to {weekly_actuals['week_start'].max()}")print(f"Weeks: {weekly_actuals['week_start'].nunique()}")print(f"Entity-Liq combinations: {weekly_actuals.groupby(['entity_id', 'liquidity_group']).ngroups}")print(f"\nColumns: {list(weekly_actuals.columns)}")print("\nDaily pattern features included:")daily_pattern_cols = [col for col in weekly_actuals.columns if any(x in col.lower() for x in ['weekend', 'monday', 'friday', 'early', 'late', 'month', 'pct'])]for col in daily_pattern_cols:    print(f"  - {col}")print("\nFirst 10 rows:")weekly_actuals.head(10)

In [ ]:
# Visualize weekly actuals
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Total weekly cash flow
weekly_total = weekly_actuals.groupby('week_start')['amount_eur'].sum()
axes[0].plot(weekly_total.index, weekly_total.values, marker='o', markersize=3)
axes[0].set_xlabel('Week Start')
axes[0].set_ylabel('Total Amount (EUR)')
axes[0].set_title('Total Weekly Cash Flow (All Entities)')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0].grid(True, alpha=0.3)

# Weekly amount distribution
axes[1].hist(weekly_actuals['amount_eur'], bins=50, edgecolor='black')
axes[1].set_xlabel('Weekly Amount (EUR)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Weekly Amounts')
axes[1].axvline(0, color='red', linestyle='--', label='Zero line')
axes[1].legend()

plt.tight_layout()
plt.show()

# Statistics
print("Weekly Actuals Statistics:")
print(f"  Mean: €{weekly_actuals['amount_eur'].mean():,.2f}")
print(f"  Median: €{weekly_actuals['amount_eur'].median():,.2f}")
print(f"  Std: €{weekly_actuals['amount_eur'].std():,.2f}")
print(f"  Min: €{weekly_actuals['amount_eur'].min():,.2f}")
print(f"  Max: €{weekly_actuals['amount_eur'].max():,.2f}")
print(f"  % Negative: {(weekly_actuals['amount_eur'] < 0).sum() / len(weekly_actuals) * 100:.1f}%")

## 7. Pivot LP ForecastsTransform LP forecasts from long format (forecast_date, horizon, amount) to wide format (W1_Forecast, W2_Forecast, etc.).

In [ ]:
# Check LP forecast structure
print("LP Forecasts Structure:")
print(lp_clean.head(20))
print(f"\nUnique horizons: {sorted(lp_clean['horizon'].unique())}")

In [ ]:
# Pivot LP forecasts
# Rename forecast_date to week_start for consistency
lp_clean['week_start'] = lp_clean['forecast_date']

# Pivot: one row per (entity, liquidity_group, week_start) with columns W1_Forecast, W2_Forecast, etc.
lp_pivoted = lp_clean.pivot_table(
    index=['entity_id', 'liquidity_group', 'week_start'],
    columns='horizon',
    values='forecast_amount_eur',
    aggfunc='first'  # In case of duplicates, take first
).reset_index()

# Rename columns
lp_pivoted.columns = ['entity_id', 'liquidity_group', 'week_start'] + \
                     [f'W{h}_Forecast' for h in range(1, 9) if h in lp_pivoted.columns[3:]]

print("=" * 80)
print("LP FORECASTS (Pivoted)")
print("=" * 80)
print(f"Shape: {lp_pivoted.shape}")
print(f"Columns: {list(lp_pivoted.columns)}")
print("\nFirst 10 rows:")
lp_pivoted.head(10)

In [ ]:
# Check for zeros in LP forecasts
forecast_cols = [col for col in lp_pivoted.columns if col.endswith('_Forecast')]
print("Zeros in LP Forecast Columns:")
for col in forecast_cols:
    if col in lp_pivoted.columns:
        zero_count = (lp_pivoted[col] == 0).sum()
        total = len(lp_pivoted)
        print(f"  {col}: {zero_count} ({zero_count/total*100:.1f}%)")

## 8. Merge Actuals and LP Forecasts

In [ ]:
# Merge weekly actuals with LP forecasts
merged_df = weekly_actuals.merge(
    lp_pivoted,
    on=['entity_id', 'liquidity_group', 'week_start'],
    how='left'  # Keep all actuals, even if no LP forecast
)

print("=" * 80)
print("MERGED DATA (Actuals + LP Forecasts)")
print("=" * 80)
print(f"Shape: {merged_df.shape}")
print(f"Date range: {merged_df['week_start'].min()} to {merged_df['week_start'].max()}")
print(f"\nColumns: {list(merged_df.columns)}")
print("\nMissing values:")
print(merged_df.isnull().sum())
print("\nFirst 10 rows:")
merged_df.head(10)

## 9. Feature EngineeringCreate features:1. **Lag Features**: Previous week values (lag_1 to lag_52)2. **Rolling Features**: Moving averages, std, min, max3. **Calendar Features**: Month, quarter, week of year, day of week4. **Statistical Features**: Exponential smoothing, volatility5. **LP Forecast Features**: Already included from merge

In [ ]:
# Sort data for feature engineering
merged_df = merged_df.sort_values(['entity_id', 'liquidity_group', 'week_start']).reset_index(drop=True)

# Initialize features dataframe
features_df = merged_df.copy()

print("Starting Feature Engineering...")
print(f"Initial shape: {features_df.shape}")

In [ ]:
# 1. LAG FEATURES (lag_1 to lag_52)
print("\n1. Creating Lag Features (lag_1 to lag_52)...")

for lag in range(1, 53):
    features_df[f'lag_{lag}'] = features_df.groupby(['entity_id', 'liquidity_group'])['amount_eur'].shift(lag)
    if lag % 10 == 0:
        print(f"  Created lag_{lag}")

print(f"  Created 52 lag features")

In [ ]:
# 2. ROLLING FEATURES
print("\n2. Creating Rolling Features...")

windows = [4, 8, 12, 26, 52]  # 1-month, 2-month, 3-month, 6-month, 1-year

for window in windows:
    # Rolling mean
    features_df[f'rolling_{window}w_mean'] = features_df.groupby(
        ['entity_id', 'liquidity_group']
    )['amount_eur'].transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    
    # Rolling std
    features_df[f'rolling_{window}w_std'] = features_df.groupby(
        ['entity_id', 'liquidity_group']
    )['amount_eur'].transform(lambda x: x.shift(1).rolling(window, min_periods=1).std())
    
    # Rolling min
    features_df[f'rolling_{window}w_min'] = features_df.groupby(
        ['entity_id', 'liquidity_group']
    )['amount_eur'].transform(lambda x: x.shift(1).rolling(window, min_periods=1).min())
    
    # Rolling max
    features_df[f'rolling_{window}w_max'] = features_df.groupby(
        ['entity_id', 'liquidity_group']
    )['amount_eur'].transform(lambda x: x.shift(1).rolling(window, min_periods=1).max())
    
    print(f"  Created rolling features for window={window} weeks")

print(f"  Created {len(windows) * 4} rolling features")

In [ ]:
# 3. CALENDAR FEATURES
print("\n3. Creating Calendar Features...")

features_df['month'] = features_df['week_start'].dt.month
features_df['quarter'] = features_df['week_start'].dt.quarter
features_df['week_of_year'] = features_df['week_start'].dt.isocalendar().week
features_df['day_of_week'] = features_df['week_start'].dt.dayofweek  # 0=Monday
features_df['is_month_start'] = (features_df['week_start'].dt.day <= 7).astype(int)
features_df['is_month_end'] = (features_df['week_start'].dt.day >= 21).astype(int)
features_df['is_quarter_start'] = ((features_df['month'].isin([1, 4, 7, 10])) & 
                                    (features_df['week_start'].dt.day <= 7)).astype(int)
features_df['is_quarter_end'] = ((features_df['month'].isin([3, 6, 9, 12])) & 
                                  (features_df['week_start'].dt.day >= 21)).astype(int)

# Cyclical encoding for month and week_of_year
features_df['month_sin'] = np.sin(2 * np.pi * features_df['month'] / 12)
features_df['month_cos'] = np.cos(2 * np.pi * features_df['month'] / 12)
features_df['week_sin'] = np.sin(2 * np.pi * features_df['week_of_year'] / 52)
features_df['week_cos'] = np.cos(2 * np.pi * features_df['week_of_year'] / 52)

print(f"  Created 12 calendar features")

In [ ]:
# 4. STATISTICAL FEATURES
print("\n4. Creating Statistical Features...")

# Exponential moving average (different spans)
for span in [4, 12, 26]:
    features_df[f'ema_{span}w'] = features_df.groupby(
        ['entity_id', 'liquidity_group']
    )['amount_eur'].transform(lambda x: x.shift(1).ewm(span=span, adjust=False).mean())

# Coefficient of variation (rolling)
features_df['cv_12w'] = (features_df['rolling_12w_std'] / features_df['rolling_12w_mean'].abs()).replace([np.inf, -np.inf], np.nan)

# Trend (difference from 4-week ago)
features_df['trend_4w'] = features_df.groupby(['entity_id', 'liquidity_group'])['amount_eur'].diff(4)

# Volatility (12-week rolling std of returns)
features_df['volatility_12w'] = features_df.groupby(
    ['entity_id', 'liquidity_group']
)['amount_eur'].transform(lambda x: x.pct_change().shift(1).rolling(12, min_periods=1).std())

print(f"  Created 9 statistical features")

In [ ]:
# Check feature creation
print("\n" + "=" * 80)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 80)
print(f"Final shape: {features_df.shape}")
print(f"Total features created: {features_df.shape[1] - merged_df.shape[1]}")
print(f"\nMissing values by column (top 10):")
missing = features_df.isnull().sum().sort_values(ascending=False).head(10)
print(missing)
print("\nFirst 5 rows:")
features_df.head()

## 10. EDA on Features

In [ ]:
# Feature correlation with target
feature_cols = [col for col in features_df.columns 
                if col not in ['entity_id', 'liquidity_group', 'week_start', 'amount_eur', 'transaction_count']]

# Sample data for correlation (too many rows otherwise)
sample_df = features_df.sample(min(10000, len(features_df)), random_state=42)

correlations = sample_df[feature_cols + ['amount_eur']].corr()['amount_eur'].drop('amount_eur').abs().sort_values(ascending=False)

print("Top 20 Features by Correlation with Target:")
print(correlations.head(20))

# Plot top correlations
plt.figure(figsize=(10, 8))
correlations.head(20).plot(kind='barh')
plt.xlabel('Absolute Correlation with amount_eur')
plt.title('Top 20 Features by Correlation')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Check LP forecast usage
forecast_cols = [col for col in features_df.columns if 'Forecast' in col]
print(f"\nLP Forecast Columns: {forecast_cols}")
print("\nLP Forecast Statistics:")
for col in forecast_cols:
    if col in features_df.columns:
        print(f"\n{col}:")
        print(f"  Count: {features_df[col].notna().sum()}")
        print(f"  Mean: €{features_df[col].mean():,.2f}")
        print(f"  Std: €{features_df[col].std():,.2f}")
        print(f"  Zeros: {(features_df[col] == 0).sum()}")
        print(f"  NaNs: {features_df[col].isna().sum()}")

## 11. Prepare Training DatasetTwo versions:1. **Enhanced**: Full dataset with all features2. **Clean**: Start from week 53, remove leakage features, LP zeros → NaN

In [ ]:
# ENHANCED DATASET (Original approach)
print("=" * 80)
print("ENHANCED DATASET")
print("=" * 80)

enhanced_df = features_df.copy()

# Convert features to numeric
feature_cols = [col for col in enhanced_df.columns 
                if col not in ['entity_id', 'liquidity_group', 'week_start', 'amount_eur']]

for col in feature_cols:
    enhanced_df[col] = pd.to_numeric(enhanced_df[col], errors='coerce')

# Fill NaN with 0 (original approach)
enhanced_df[feature_cols] = enhanced_df[feature_cols].fillna(0)

print(f"Shape: {enhanced_df.shape}")
print(f"Features: {len(feature_cols)}")
print(f"Date range: {enhanced_df['week_start'].min()} to {enhanced_df['week_start'].max()}")
print(f"Weeks: {enhanced_df['week_start'].nunique()}")

# Save
enhanced_df.to_csv('../data/intermediate/training_data_enhanced.csv', index=False)
print("\nSaved to: data/intermediate/training_data_enhanced.csv")

In [ ]:
# CLEAN DATASET (Conservative approach)
print("\n" + "=" * 80)
print("CLEAN DATASET")
print("=" * 80)

clean_df = features_df.copy()

# 1. Start from week 53 (Feb 27, 2023) - after lag_52 fills
cutoff_date = pd.Timestamp('2023-02-27')
clean_df = clean_df[clean_df['week_start'] >= cutoff_date].copy()
print(f"1. Filtered to week 53+: {len(clean_df)} rows")

# 2. Replace zeros in LP features with NaN
lp_features = ['W1_Forecast', 'W2_Forecast', 'W3_Forecast', 'W4_Forecast']
zeros_replaced = 0
for col in lp_features:
    if col in clean_df.columns:
        zero_count = (clean_df[col] == 0).sum()
        clean_df.loc[clean_df[col] == 0, col] = np.nan
        zeros_replaced += zero_count
print(f"2. Replaced {zeros_replaced} zeros in LP features with NaN")

# 3. Remove forecast-vs-actual comparison features (data leakage)
vs_actual_features = [col for col in clean_df.columns if 'vs_actual' in col.lower() or 'v_actual' in col.lower()]
if vs_actual_features:
    clean_df = clean_df.drop(columns=vs_actual_features)
    print(f"3. Removed {len(vs_actual_features)} forecast-vs-actual features (data leakage)")

# 4. Remove zero-variance features
zero_var_features = []
for col in clean_df.columns:
    if col not in ['entity_id', 'liquidity_group', 'week_start', 'amount_eur']:
        if clean_df[col].dtype in ['int64', 'float64']:
            if clean_df[col].nunique() <= 1:
                zero_var_features.append(col)
if zero_var_features:
    clean_df = clean_df.drop(columns=zero_var_features)
    print(f"4. Removed {len(zero_var_features)} zero-variance features: {zero_var_features}")

# 5. Convert features to numeric
clean_feature_cols = [col for col in clean_df.columns 
                      if col not in ['entity_id', 'liquidity_group', 'week_start', 'amount_eur']]
for col in clean_feature_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors='coerce')

# Keep NaN for LP features, fill 0 for others
other_features = [col for col in clean_feature_cols if col not in lp_features]
clean_df[other_features] = clean_df[other_features].fillna(0)

print(f"\nFinal shape: {clean_df.shape}")
print(f"Features: {len(clean_feature_cols)}")
print(f"Date range: {clean_df['week_start'].min()} to {clean_df['week_start'].max()}")
print(f"Weeks: {clean_df['week_start'].nunique()}")

# Save
clean_df.to_csv('../data/intermediate/training_data_clean.csv', index=False)
print("\nSaved to: data/intermediate/training_data_clean.csv")

## 12. Choose Training Dataset

In [ ]:
# SELECT WHICH DATASET TO USE FOR MODELING
# Change this to 'clean' to use clean dataset
DATASET_VERSION = 'enhanced'  # Options: 'enhanced' or 'clean'

if DATASET_VERSION == 'enhanced':
    training_data = enhanced_df.copy()
    print("Using ENHANCED dataset")
else:
    training_data = clean_df.copy()
    print("Using CLEAN dataset")

print(f"Shape: {training_data.shape}")
print(f"Date range: {training_data['week_start'].min()} to {training_data['week_start'].max()}")

## 13. Define Backtesting Framework

In [ ]:
# Walk-Forward Backtesting Configuration
VALIDATION_START = pd.Timestamp('2025-07-07')  # 3-month validation
VALIDATION_END = pd.Timestamp('2025-09-29')
HORIZONS = list(range(1, 9))  # W1 to W8

# WAPE Targets
WAPE_TARGETS = {
    1: 5.0, 2: 7.5, 3: 10.0, 4: 12.5,
    5: 15.0, 6: 17.5, 7: 20.0, 8: 22.5
}

print("Backtesting Configuration:")
print(f"  Validation Start: {VALIDATION_START.date()}")
print(f"  Validation End: {VALIDATION_END.date()}")
print(f"  Validation Weeks: {len(pd.date_range(VALIDATION_START, VALIDATION_END, freq='W-MON'))}")
print(f"  Horizons: {HORIZONS}")
print(f"  WAPE Targets: {WAPE_TARGETS}")

In [ ]:
# Helper functions for backtesting

def calculate_wape(y_true, y_pred):
    """Calculate Weighted Absolute Percentage Error"""
    return (np.abs(y_true - y_pred).sum() / np.abs(y_true).sum()) * 100

def calculate_directionality(y_true, y_pred):
    """Calculate percentage of correct direction predictions"""
    direction_actual = np.where(y_true > 0, 1, -1)
    direction_pred = np.where(y_pred > 0, 1, -1)
    return (direction_actual == direction_pred).sum() / len(y_true) * 100

def prepare_train_test(data, test_week, horizon):
    """Prepare train and test sets for a specific test week and horizon"""
    # Target week is test_week + (horizon - 1) weeks
    target_week = test_week + timedelta(weeks=horizon - 1)
    
    # Training data: everything before test_week
    train = data[data['week_start'] < test_week].copy()
    
    # Test data: specific entity-liq combinations at target_week
    test = data[data['week_start'] == test_week].copy()
    actuals = data[data['week_start'] == target_week][['entity_id', 'liquidity_group', 'amount_eur']].copy()
    actuals.columns = ['entity_id', 'liquidity_group', 'actual']
    
    test = test.merge(actuals, on=['entity_id', 'liquidity_group'], how='inner')
    
    return train, test

print("Backtesting helper functions defined")

## 14. Model Definitions### 14.1 LightGBM

In [ ]:
# LightGBM Configuration
LGB_PARAMS = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42
}

def train_lgb_model(X_train, y_train, params=None):
    """Train LightGBM model"""
    if params is None:
        params = LGB_PARAMS.copy()
    
    train_data = lgb.Dataset(X_train, label=y_train)
    model = lgb.train(params, train_data, num_boost_round=100)
    return model

print("LightGBM model functions defined")

### 14.2 XGBoost

In [ ]:
# XGBoost Configuration
XGB_PARAMS = {
    'objective': 'reg:squarederror',
    'eval_metric': 'mae',
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42,
    'verbosity': 0
}

def train_xgb_model(X_train, y_train, params=None):
    """Train XGBoost model"""
    if params is None:
        params = XGB_PARAMS.copy()
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    model = xgb.train(params, dtrain, num_boost_round=100)
    return model

print("XGBoost model functions defined")

### 14.3 Strategy DefinitionsThree forecasting strategies:1. **Recursive**: Train one model, use its predictions recursively for future horizons2. **Direct**: Train separate model for each horizon3. **DirRec**: Direct for W1-W4, Recursive for W5-W8

In [ ]:
def recursive_forecast(model, X_test, horizons, model_type='lgb'):
    """
    Recursive forecasting: use model's own predictions as inputs for future horizons
    """
    forecasts = {}
    X_current = X_test.copy()
    
    for h in horizons:
        # Predict
        if model_type == 'lgb':
            pred = model.predict(X_current)
        elif model_type == 'xgb':
            dtest = xgb.DMatrix(X_current)
            pred = model.predict(dtest)
        else:
            raise ValueError(f"Unknown model type: {model_type}")
        
        forecasts[h] = pred
        
        # Update features for next horizon (shift lags)
        if h < max(horizons):
            # Shift lag features
            lag_cols = [col for col in X_current.columns if col.startswith('lag_')]
            for i in range(len(lag_cols) - 1, 0, -1):
                X_current[f'lag_{i+1}'] = X_current[f'lag_{i}']
            if 'lag_1' in X_current.columns:
                X_current['lag_1'] = pred
    
    return forecasts

print("Recursive forecasting strategy defined")

## 15. Run Backtests### 15.1 LightGBM + Recursive

In [ ]:
# LightGBM Recursive Backtest
print("=" * 80)
print("LIGHTGBM + RECURSIVE BACKTEST")
print("=" * 80)

# Prepare data
feature_cols = [col for col in training_data.columns 
                if col not in ['entity_id', 'liquidity_group', 'week_start', 'amount_eur']]

validation_weeks = pd.date_range(VALIDATION_START, VALIDATION_END, freq='W-MON')
results_lgb_recursive = []

print(f"Features: {len(feature_cols)}")
print(f"Validation weeks: {len(validation_weeks)}")
print(f"Starting backtest...\n")

for i, test_week in enumerate(validation_weeks):
    print(f"Week {i+1}/{len(validation_weeks)}: {test_week.date()}", end=' ... ')
    
    # Prepare training data (all data before test_week)
    train_data = training_data[training_data['week_start'] < test_week].copy()
    test_data = training_data[training_data['week_start'] == test_week].copy()
    
    if len(train_data) == 0 or len(test_data) == 0:
        print("Skipped (no data)")
        continue
    
    # Group by entity-liq
    for (entity_id, liquidity_group), group in test_data.groupby(['entity_id', 'liquidity_group']):
        # Train model on this entity-liq's historical data
        entity_train = train_data[
            (train_data['entity_id'] == entity_id) & 
            (train_data['liquidity_group'] == liquidity_group)
        ]
        
        if len(entity_train) < 10:  # Need minimum training data
            continue
        
        X_train = entity_train[feature_cols]
        y_train = entity_train['amount_eur']
        X_test = group[feature_cols]
        
        # Train model
        try:
            model = train_lgb_model(X_train, y_train)
            
            # Recursive forecasts for W1-W8
            forecasts = recursive_forecast(model, X_test, HORIZONS, model_type='lgb')
            
            # Get actuals for each horizon
            for h in HORIZONS:
                target_week = test_week + timedelta(weeks=h - 1)
                actual_row = training_data[
                    (training_data['week_start'] == target_week) &
                    (training_data['entity_id'] == entity_id) &
                    (training_data['liquidity_group'] == liquidity_group)
                ]
                
                if len(actual_row) > 0:
                    actual = actual_row['amount_eur'].values[0]
                    forecast = forecasts[h][0]
                    
                    results_lgb_recursive.append({
                        'forecast_date': test_week,
                        'horizon': h,
                        'target_date': target_week,
                        'entity_id': entity_id,
                        'liquidity_group': liquidity_group,
                        'model': 'lightgbm',
                        'strategy': 'recursive',
                        'actual': actual,
                        'forecast_p90': forecast,
                        'abs_error': abs(actual - forecast),
                        'ape': abs(actual - forecast) / abs(actual) if actual != 0 else 0
                    })
        except Exception as e:
            continue
    
    print("Done")

results_lgb_recursive_df = pd.DataFrame(results_lgb_recursive)
print(f"\nBacktest complete. Total forecasts: {len(results_lgb_recursive_df)}")

## 16. Evaluation Metrics

In [ ]:
# Filter near-zero actuals for more reliable WAPE
results_filtered = results_lgb_recursive_df[results_lgb_recursive_df['actual'].abs() >= 1000].copy()

print("=" * 80)
print("LIGHTGBM RECURSIVE - EVALUATION METRICS")
print("=" * 80)
print(f"Total forecasts: {len(results_lgb_recursive_df):,}")
print(f"After filtering |actual| < €1,000: {len(results_filtered):,}")
print()

In [ ]:
# Portfolio WAPE by Horizon
print("Portfolio WAPE by Horizon:")
print("-" * 80)
print(f"{'Horizon':<10} {'WAPE':<10} {'Target':<10} {'Gap':<10} {'Status':<10}")
print("-" * 80)

wapes_by_horizon = {}
for h in HORIZONS:
    h_data = results_filtered[results_filtered['horizon'] == h]
    if len(h_data) > 0:
        total_actual = h_data['actual'].abs().sum()
        total_error = h_data['abs_error'].sum()
        wape = (total_error / total_actual * 100) if total_actual > 0 else 0
        wapes_by_horizon[h] = wape
        
        target = WAPE_TARGETS[h]
        gap = wape - target
        status = "✅ PASS" if wape <= target else "❌ FAIL"
        
        print(f"W{h:<9} {wape:>6.2f}%    ≤{target:>4.1f}%    {gap:>+6.2f}%   {status:<10}")

print()

In [ ]:
# Directionality
results_filtered['direction_actual'] = np.where(results_filtered['actual'] > 0, 1, -1)
results_filtered['direction_forecast'] = np.where(results_filtered['forecast_p90'] > 0, 1, -1)
results_filtered['direction_correct'] = (results_filtered['direction_actual'] == results_filtered['direction_forecast']).astype(int)
directionality = results_filtered['direction_correct'].mean() * 100

print(f"Overall Directionality: {directionality:.2f}%")
print()

In [ ]:
# Comparison to Treasury LP Baseline
treasury_lp_wape = 10.0
ml_w1_wape = wapes_by_horizon.get(1, 0)
gap_to_lp = ml_w1_wape - treasury_lp_wape

print("Comparison to Treasury LP Baseline:")
print("-" * 80)
print(f"Treasury LP W1 WAPE:       {treasury_lp_wape:.2f}%")
print(f"ML W1 WAPE:                {ml_w1_wape:.2f}%")
print(f"Gap:                       {gap_to_lp:+.2f} pp")
print()

## 17. Visualizations

In [ ]:
# Plot 1: WAPE by Horizon
fig, ax = plt.subplots(figsize=(12, 6))

horizons = list(wapes_by_horizon.keys())
wapes = list(wapes_by_horizon.values())
targets = [WAPE_TARGETS[h] for h in horizons]

x = np.arange(len(horizons))
width = 0.35

bars1 = ax.bar(x - width/2, wapes, width, label='Actual WAPE', color='steelblue')
bars2 = ax.bar(x + width/2, targets, width, label='Target WAPE', color='lightcoral')

ax.set_xlabel('Horizon')
ax.set_ylabel('WAPE (%)')
ax.set_title('Portfolio WAPE by Horizon vs Targets')
ax.set_xticks(x)
ax.set_xticklabels([f'W{h}' for h in horizons])
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Entity Performance (W1 WAPE)
w1_data = results_filtered[results_filtered['horizon'] == 1]
entity_perf = []

for (entity_id, liquidity_group), group in w1_data.groupby(['entity_id', 'liquidity_group']):
    total_actual = group['actual'].abs().sum()
    total_error = group['abs_error'].sum()
    wape = (total_error / total_actual * 100) if total_actual > 0 else 0
    
    entity_perf.append({
        'combo': f"{entity_id}-{liquidity_group}",
        'wape': wape,
        'total_actual': total_actual
    })

entity_df = pd.DataFrame(entity_perf).sort_values('wape')

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
colors = ['green' if w <= 5.0 else 'orange' if w <= 10.0 else 'red' for w in entity_df['wape']]
ax.barh(entity_df['combo'], entity_df['wape'], color=colors)
ax.axvline(5.0, color='green', linestyle='--', label='Target (5%)', alpha=0.7)
ax.axvline(10.0, color='orange', linestyle='--', label='LP Baseline (10%)', alpha=0.7)
ax.set_xlabel('W1 WAPE (%)')
ax.set_ylabel('Entity-Liquidity')
ax.set_title('W1 WAPE by Entity-Liquidity Combination')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nTop 5 Performers (W1 WAPE):")
print(entity_df.head())
print("\nBottom 5 Performers (W1 WAPE):")
print(entity_df.tail())

In [ ]:
# Plot 3: Forecast vs Actual (Sample entity, W1)
# Pick entity with most data
w1_sample = w1_data.groupby(['entity_id', 'liquidity_group']).size().sort_values(ascending=False).head(1)
if len(w1_sample) > 0:
    sample_entity = w1_sample.index[0][0]
    sample_liq = w1_sample.index[0][1]
    
    sample_data = w1_data[
        (w1_data['entity_id'] == sample_entity) & 
        (w1_data['liquidity_group'] == sample_liq)
    ].sort_values('forecast_date')
    
    fig, ax = plt.subplots(figsize=(15, 6))
    ax.plot(sample_data['forecast_date'], sample_data['actual'], marker='o', label='Actual', linewidth=2)
    ax.plot(sample_data['forecast_date'], sample_data['forecast_p90'], marker='s', label='Forecast', linewidth=2, alpha=0.7)
    ax.set_xlabel('Forecast Date')
    ax.set_ylabel('Amount (EUR)')
    ax.set_title(f'W1 Forecast vs Actual: {sample_entity}-{sample_liq}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='black', linestyle='-', linewidth=0.5)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot 4: Error Distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# APE distribution
axes[0].hist(results_filtered['ape'].clip(0, 1), bins=50, edgecolor='black')
axes[0].set_xlabel('Absolute Percentage Error')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Absolute Percentage Error (clipped at 100%)')
axes[0].axvline(0.05, color='green', linestyle='--', label='5% (W1 target)', alpha=0.7)
axes[0].axvline(0.10, color='orange', linestyle='--', label='10% (LP baseline)', alpha=0.7)
axes[0].legend()

# Absolute error distribution
axes[1].hist(results_filtered['abs_error'], bins=50, edgecolor='black')
axes[1].set_xlabel('Absolute Error (EUR)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Absolute Error')

plt.tight_layout()
plt.show()

## 18. Summary and Conclusions

In [ ]:
print("=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"\nDataset: {DATASET_VERSION.upper()}")
print(f"Model: LightGBM")
print(f"Strategy: Recursive")
print(f"Validation Period: {VALIDATION_START.date()} to {VALIDATION_END.date()}")
print(f"Total Forecasts: {len(results_lgb_recursive_df):,}")
print(f"Filtered Forecasts: {len(results_filtered):,}")
print()
print("Key Metrics:")
print(f"  W1 WAPE: {wapes_by_horizon.get(1, 0):.2f}% (Target: 5.0%)")
print(f"  Gap to LP Baseline: {gap_to_lp:+.2f} pp")
print(f"  Directionality: {directionality:.2f}%")
print()
print("Horizons Passing Target:")
passing = [f"W{h}" for h in HORIZONS if wapes_by_horizon.get(h, 100) <= WAPE_TARGETS[h]]
print(f"  {len(passing)}/{len(HORIZONS)}: {', '.join(passing) if passing else 'None'}")
print()
print("Key Issues Identified:")
if gap_to_lp > 5:
    print("  ⚠️  ML significantly worse than Treasury LP baseline")
if wapes_by_horizon.get(1, 0) > 15:
    print("  ⚠️  W1 WAPE far from target (>15%)")
if directionality < 70:
    print("  ⚠️  Poor directionality (<70%)")
else:
    print("  ✅ Excellent directionality (>95%)")
print()
print("Next Steps:")
print("  1. Investigate why ML can't match LP baseline (10% vs ~20%)")
print("  2. Check if LP forecasts (W1-W4) are being properly used as features")
print("  3. Analyze feature importance to understand model's decision-making")
print("  4. Try different models (XGBoost, LSTM, SARIMAX) and strategies (Direct, DirRec)")
print("  5. Examine entity-level differences (why do big entities perform better?)")
print("=" * 80)

## 19. Feature Importance Analysis

In [ ]:
# Train a final model to analyze feature importance
print("Training final model for feature importance analysis...")

# Use all training data before validation period
train_all = training_data[training_data['week_start'] < VALIDATION_START].copy()
X_train_all = train_all[feature_cols]
y_train_all = train_all['amount_eur']

final_model = train_lgb_model(X_train_all, y_train_all)

# Get feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': final_model.feature_importance()
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features:")
print(importance.head(20))

# Plot
plt.figure(figsize=(10, 8))
plt.barh(range(20), importance.head(20)['importance'])
plt.yticks(range(20), importance.head(20)['feature'])
plt.xlabel('Importance')
plt.title('Top 20 Feature Importance (LightGBM)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Check if LP forecasts are in top features
lp_in_top20 = importance.head(20)['feature'].str.contains('Forecast').sum()
print(f"\nLP Forecast features in top 20: {lp_in_top20}")
if lp_in_top20 == 0:
    print("⚠️  WARNING: No LP forecast features in top 20 - model not leveraging Treasury forecasts!")

## 20. Investigation: LP Forecast Usage

In [ ]:
# Detailed analysis of LP forecast features
print("=" * 80)
print("LP FORECAST FEATURE ANALYSIS")
print("=" * 80)

lp_features = [col for col in feature_cols if 'Forecast' in col]
print(f"\nLP Forecast features: {lp_features}")
print()

for col in lp_features:
    if col in training_data.columns:
        print(f"{col}:")
        print(f"  Non-null count: {training_data[col].notna().sum()} ({training_data[col].notna().sum()/len(training_data)*100:.1f}%)")
        print(f"  Zeros: {(training_data[col] == 0).sum()}")
        print(f"  NaNs: {training_data[col].isna().sum()}")
        print(f"  Mean: €{training_data[col].mean():,.2f}")
        print(f"  Std: €{training_data[col].std():,.2f}")
        
        # Find importance rank
        if col in importance['feature'].values:
            rank = importance[importance['feature'] == col].index[0] + 1
            imp_value = importance[importance['feature'] == col]['importance'].values[0]
            print(f"  Importance rank: {rank}/{len(feature_cols)} (importance: {imp_value:.0f})")
        print()

# Correlation between LP forecasts and actual
print("\nCorrelation between LP forecasts and actuals:")
for col in lp_features:
    if col in training_data.columns:
        corr = training_data[[col, 'amount_eur']].corr().iloc[0, 1]
        print(f"  {col} vs amount_eur: {corr:.3f}")

## 21. Save Results

In [ ]:
# Save backtest results
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_dir = Path('../artifacts/backtesting/notebook')
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / f'lgb_recursive_{DATASET_VERSION}_{timestamp}.csv'
results_lgb_recursive_df.to_csv(output_file, index=False)

print(f"Results saved to: {output_file}")
print(f"Total rows: {len(results_lgb_recursive_df):,}")

---## End of Notebook**Next steps for investigation**:1. Run this notebook and examine results cell by cell2. Check if LP forecasts are being properly used (Section 19)3. Try different dataset versions (enhanced vs clean)4. Experiment with other models (XGBoost, LSTM, SARIMAX)5. Test different strategies (Direct, DirRec)6. Analyze entity-specific patterns7. Investigate why ML can't match LP baseline (10% WAPE)